# 04 - Forecasting and Capacity Planning

This notebook explores capacity exhaustion forecasts, latency forecasts, and what-if capacity planning.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.control_plane.inference_hub import InferenceHub
from src.control_plane.simulator import WhatIfSimulator
from src.models.forecasting.nbeats_model import forecast_volume

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "io_features.parquet"
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
latest_ts = df["timestamp"].max()

hub = InferenceHub(project_root=PROJECT_ROOT)
simulator = WhatIfSimulator(hub)
print("Latest timestamp:", latest_ts)
print("InferenceHub and WhatIfSimulator initialized.")

In [ ]:
# Pick six volumes: 2 high, 2 medium, 2 low capacity used pct
latest_rows = df.sort_values("timestamp").groupby("volume_id", as_index=False).tail(1).copy()
latest_rows = latest_rows.sort_values("capacity_used_pct", ascending=False).reset_index(drop=True)
high_vols = latest_rows.head(2)["volume_id"].tolist()
low_vols = latest_rows.tail(2)["volume_id"].tolist()
mid_start = max(0, len(latest_rows) // 2 - 1)
mid_vols = latest_rows.iloc[mid_start:mid_start + 2]["volume_id"].tolist()
selected_vols = high_vols + mid_vols + low_vols
display(latest_rows[latest_rows["volume_id"].isin(selected_vols)][["volume_id", "capacity_used_pct", "tier", "node_id", "workload_type"]])

analysis_rows = []
for volume_id in sorted(df["volume_id"].drop_duplicates().tolist()):
    analysis = hub.analyze_volume(volume_id, latest_ts)
    analysis_rows.append({
        "volume_id": volume_id,
        "workload_type": analysis["workload_type"],
        "hotspot_score": analysis["hotspot_score"],
        "warning_85pct_days": analysis["days_to_fill"]["warning_85pct_days"],
        "critical_95pct_days": analysis["days_to_fill"]["critical_95pct_days"],
        "bandwidth_forecast_24h": analysis["bandwidth_forecast_24h"],
        "analysis": analysis,
    })
forecast_df = pd.DataFrame(analysis_rows)
display(forecast_df[["volume_id", "workload_type", "hotspot_score", "warning_85pct_days", "critical_95pct_days"]].head())

In [ ]:
# DTF ranking for all 50 volumes
dtf_plot = forecast_df.copy()
dtf_plot["urgency"] = dtf_plot["critical_95pct_days"].fillna(9999)
dtf_plot = dtf_plot.sort_values(["urgency", "warning_85pct_days"], ascending=[True, True]).reset_index(drop=True)
y = np.arange(len(dtf_plot))
plt.figure(figsize=(14, 12))
plt.barh(y - 0.2, dtf_plot["warning_85pct_days"].fillna(0), height=0.4, label="warning_85pct_days", color="#ffcc66")
plt.barh(y + 0.2, dtf_plot["critical_95pct_days"].fillna(0), height=0.4, label="critical_95pct_days", color="#cc6666")
plt.yticks(y, dtf_plot["volume_id"])
plt.gca().invert_yaxis()
plt.xlabel("days to fill")
plt.title("Days-to-Fill Ranking Across All Volumes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# N-BEATS capacity forecast curves for 3 volumes
curve_volumes = dtf_plot.head(3)["volume_id"].tolist()
fig, axes = plt.subplots(nrows=len(curve_volumes), ncols=1, figsize=(14, 10), sharex=False)
if len(curve_volumes) == 1:
    axes = [axes]

for ax, volume_id in zip(axes, curve_volumes):
    history = hub.get_nbeats_input(volume_id, latest_ts)
    forecast = forecast_volume(hub.nbeats, history, n_steps_ahead=60, device="cpu")
    history_pct = history * 100.0
    forecast_pct = np.clip(forecast, 0.0, 1.0) * 100.0
    hist_x = np.arange(-len(history_pct) + 1, 1)
    fc_x = np.arange(1, len(forecast_pct) + 1)
    ax.plot(hist_x, history_pct, marker="o", label="historical")
    ax.plot(fc_x, forecast_pct, marker="o", label="forecast")
    ax.axhline(85, linestyle="--", color="#ffcc66", label="85% threshold")
    ax.axhline(95, linestyle="--", color="#cc6666", label="95% threshold")
    ax.set_title(volume_id)
    ax.set_ylabel("capacity_used_pct")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("days relative to latest timestamp")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# TFT latency forecast fan chart and what-if simulation
latency_volumes = dtf_plot.head(3)["volume_id"].tolist()
fig, axes = plt.subplots(nrows=len(latency_volumes), ncols=1, figsize=(14, 10), sharex=True)
if len(latency_volumes) == 1:
    axes = [axes]

for ax, volume_id in zip(axes, latency_volumes):
    row = forecast_df[forecast_df["volume_id"] == volume_id].iloc[0]
    forecast_24h = row["bandwidth_forecast_24h"]
    p50 = np.array(forecast_24h["p50_latency_us"])
    p90 = np.array(forecast_24h["p90_latency_us"])
    p95 = np.array(forecast_24h["p95_latency_us"])
    steps = np.arange(1, len(p50) + 1)
    ax.plot(steps, p50, label="p50", color="tab:blue")
    ax.plot(steps, p90, label="p90", color="tab:orange", linestyle="--")
    ax.plot(steps, p95, label="p95", color="tab:red", linestyle="--")
    ax.fill_between(steps, p50, p95, color="tab:blue", alpha=0.15, label="p50-p95 band")
    ax.axhline(8000, color="black", linestyle=":", label="latency risk threshold")
    ax.set_title(volume_id)
    ax.set_ylabel("latency_us")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("forecast hour")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

# What-if simulation for the most urgent DTF volume
most_urgent = dtf_plot.iloc[0]["volume_id"]
sim_500 = simulator.simulate_add_capacity_scenario(most_urgent, 500.0)
sim_1000 = simulator.simulate_add_capacity_scenario(most_urgent, 1000.0)
what_if_df = pd.DataFrame([
    {"scenario": "baseline", "volume_id": most_urgent, "original_dtf_days": sim_500["original_dtf_days"], "simulated_dtf_days": sim_500["original_dtf_days"], "improvement_days": 0.0},
    {"scenario": "+500GB", "volume_id": most_urgent, "original_dtf_days": sim_500["original_dtf_days"], "simulated_dtf_days": sim_500["simulated_dtf_days"], "improvement_days": sim_500["improvement_days"]},
    {"scenario": "+1000GB", "volume_id": most_urgent, "original_dtf_days": sim_1000["original_dtf_days"], "simulated_dtf_days": sim_1000["simulated_dtf_days"], "improvement_days": sim_1000["improvement_days"]},
])
display(what_if_df)

In [ ]:
# Automated recommendations following the API logic
def build_recommendations(frame: pd.DataFrame) -> pd.DataFrame:
    recs = []
    prio_order = {"CRITICAL": 0, "HIGH": 1, "WARNING": 2, "MEDIUM": 3}
    for _, row in frame.iterrows():
        vol_id = row["volume_id"]
        analysis = row["analysis"]
        dtf = analysis.get("days_to_fill", {})
        crit_days = dtf.get("critical_95pct_days")
        warn_days = dtf.get("warning_85pct_days")
        score = analysis.get("hotspot_score", 0.0)
        risk_score = analysis.get("latency_risk_score", 0.0)

        if crit_days is not None and crit_days < 7.0:
            recs.append({
                "volume_id": vol_id,
                "priority": "CRITICAL",
                "message": f"Volume will breach capacity limit in {crit_days} days. Expand capacity or migrate cold data immediately.",
            })
        elif warn_days is not None and warn_days < 30.0:
            recs.append({
                "volume_id": vol_id,
                "priority": "WARNING",
                "message": f"Volume will exceed 85% capacity threshold in {warn_days} days. Plan expansion.",
            })

        if score >= 80:
            recs.append({
                "volume_id": vol_id,
                "priority": "HIGH",
                "message": f"Active hotspot detected (score: {score}) on volume. Severe performance anomaly.",
            })

        if risk_score >= 0.8:
            recs.append({
                "volume_id": vol_id,
                "priority": "MEDIUM",
                "message": f"High risk of SLO latency breach (peak probability: {round(risk_score * 100, 1)}%) in next 24h.",
            })

    recs_df = pd.DataFrame(recs)
    if not recs_df.empty:
        recs_df["priority_rank"] = recs_df["priority"].map(prio_order)
        recs_df = recs_df.sort_values(["priority_rank", "volume_id"]).drop(columns=["priority_rank"])
    return recs_df

recommendations_df = build_recommendations(forecast_df)
display(recommendations_df if not recommendations_df.empty else pd.DataFrame(columns=["volume_id", "priority", "message"]))